# FGVC-Aircraft — Manufacturer Classifier (ResNet34) with Full Test-Set Evaluation

This notebook builds a complete **manufacturer classification pipeline** on the **FGVC-Aircraft** dataset.

It covers the full workflow end-to-end:
- dataset download and integrity checks
- caching and extraction
- loading labels and bounding boxes
- image preprocessing
- FastAI DataLoaders
- progressive training
- full evaluation on the official test split
- export of reports and deployment artifacts

## Dataset protocol

The notebook follows the official split logic:
- **Train split**: model fitting
- **Validation split**: training monitoring and model selection
- **Test split**: final performance measurement

## Why this structure matters

Keeping train, validation, and test separate is critical at this stage:
- the validation split is still needed to monitor the training dynamics,
- the test split is reserved for the final unbiased evaluation,
- once the pipeline is stable, a final `train + val` production run can be considered later.

## Evaluation outputs

The notebook computes and exports:
- accuracy
- balanced accuracy / macro recall
- macro precision
- macro F1-score
- top-3 accuracy
- per-class performance table
- row-normalized confusion matrix
- most frequent confusion pairs
- per-image test predictions with top-5 outputs

## Important implementation detail

The training pipeline removes the 20-pixel copyright banner and crops the aircraft using the official bounding boxes.
This is deliberate: using the same image logic consistently is essential for obtaining a model that behaves correctly in deployment.


In [ ]:
from pathlib import Path
import subprocess

# =============================================================================
# DATASET CONFIGURATION
# =============================================================================
# DATA_URL:
#   Official download link for the FGVC-Aircraft 2013b archive.
#   This version includes the test annotations.
DATA_URL = r"""https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz"""

# CACHE_DIR:
#   Root cache folder inside the Kaggle working directory.
#   The dataset archive, extracted files, and preprocessed images are stored here.
CACHE_DIR = Path("/kaggle/working/datasets/fgvc-aircraft-2013b")

# TAR_PATH:
#   Full path to the downloaded .tar.gz archive.
TAR_PATH = CACHE_DIR / "fgvc-aircraft-2013b.tar.gz"

# EXTRACT_DIR:
#   Destination folder for the extracted dataset.
EXTRACT_DIR = CACHE_DIR / "extracted"

# PROC_DIR and PROC_IMAGES_DIR:
#   Folder used to store preprocessed aircraft crops.
#   Preprocessing is done once and saved on disk to avoid repeating the same
#   expensive operations at every training run.
PROC_DIR = CACHE_DIR / "processed"
PROC_IMAGES_DIR = PROC_DIR / "images_bbox_nobanner"
PROC_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# OUTPUT STRUCTURE
#   MODEL_DIR stores trained model artifacts.
OUTPUT_DIR = Path("/kaggle/working/output")
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# MODEL_PATH:
#   Final exported FastAI learner.
MODEL_PATH = MODEL_DIR / "fgvc_manufacturer_resnet34_optimized_fastai.pkl"


def _run(cmd, check=True):
    """Run a shell command and print its combined stdout/stderr.

    Parameters
    ----------
    cmd : list[str]
        Shell command represented as a list.
    check : bool
        If True, raise an exception when the command fails.

    Returns
    -------
    int
        Process return code.
    """
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {' '.join(cmd)}")
    return p.returncode


def _head_bytes(path: Path, n=300):
    """Read the first bytes of a file.

    This is useful when a supposed archive is actually an HTML error page
    or a truncated file. Printing the first bytes helps diagnose the issue.
    """
    try:
        return path.open("rb").read(n)
    except Exception as e:
        return f"<could not read bytes: {e}>"


def validate_tarball(tar_path: Path):
    """Validate that the cached archive is a real and readable .tar.gz file.

    The validation logic checks:
    1. file existence,
    2. minimum size,
    3. gzip integrity,
    4. tar listing readability.
    """
    if not tar_path.exists():
        raise RuntimeError("Tarball missing after download.")

    size = tar_path.stat().st_size
    print(f"Downloaded file size: {size/1e6:.2f} MB")

    # Very small files are usually HTML error pages or broken downloads.
    if size < 5_000_000:
        print("First bytes:", _head_bytes(tar_path))
        raise RuntimeError("Downloaded file too small; likely an HTML error page or a truncated download.")

    # gzip -t checks compressed file integrity without extracting it.
    _run(["bash", "-lc", f"gzip -t '{tar_path}'"])

    # tar -tzf lists the archive contents to confirm the tar structure is readable.
    _run(["bash", "-lc", f"tar -tzf '{tar_path}' | head"])


def download_tarball(force_redownload: bool = False):
    """Download the dataset archive unless a usable cached copy already exists.

    Parameters
    ----------
    force_redownload : bool
        If True, delete the existing archive before downloading again.
    """
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if force_redownload and TAR_PATH.exists():
        print(f"Removing corrupted or outdated tarball: {TAR_PATH}")
        TAR_PATH.unlink()

    if TAR_PATH.exists() and TAR_PATH.stat().st_size > 0:
        print(f"Tarball already present: {TAR_PATH}")
        return

    print("Downloading the FGVC-Aircraft archive with curl...")
    rc = _run(
        ["bash", "-lc", f"curl -fL --retry 5 --retry-delay 2 --connect-timeout 20 --max-time 0 -o '{TAR_PATH}' '{DATA_URL}'"],
        check=False
    )

    # wget is used as a fallback in case curl fails.
    if rc != 0:
        print("curl failed; trying wget instead...")
        if TAR_PATH.exists():
            TAR_PATH.unlink()
        _run(["bash", "-lc", f"wget -t 5 -T 20 -O '{TAR_PATH}' '{DATA_URL}'"])


def extract_tarball():
    """Extract the dataset archive only once.

    If a previous extraction already contains the expected metadata file,
    extraction is skipped.
    """
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if any(EXTRACT_DIR.rglob("images_box.txt")):
        print("Dataset extraction already exists.")
        return

    print("Extracting the archive...")
    _run(["bash", "-lc", f"tar -xzf '{TAR_PATH}' -C '{EXTRACT_DIR}' --no-same-owner"])


def locate_fgvc_data_dir(extract_dir: Path) -> Path:
    """Locate the official FGVC 'data' directory.

    The function identifies the folder containing:
    - images_box.txt
    - the split files
    - the images/ folder

    Returns
    -------
    Path
        Path to the official FGVC data directory.
    """
    hits = list(extract_dir.rglob("images_box.txt"))
    if not hits:
        raise RuntimeError("Could not locate images_box.txt after extraction.")

    data_dir = hits[0].parent

    if not (data_dir / "images").exists():
        raise RuntimeError(f"Found images_box.txt in {data_dir}, but the images/ directory is missing.")

    return data_dir


def ensure_fgvc_ready():
    """Return a ready-to-use FGVC data directory.

    This function:
    1. reuses a valid extracted cache when available,
    2. downloads the archive if missing,
    3. validates the archive,
    4. deletes and re-downloads it once if corrupted,
    5. extracts the archive,
    6. returns the official data directory.
    """
    if EXTRACT_DIR.exists() and any(EXTRACT_DIR.rglob("images_box.txt")):
        return locate_fgvc_data_dir(EXTRACT_DIR)

    download_tarball()

    try:
        validate_tarball(TAR_PATH)
    except Exception as e:
        print("Cached tarball validation failed. The file will be deleted and downloaded again once.")
        print("Validation error:", e)
        if TAR_PATH.exists():
            TAR_PATH.unlink()
        download_tarball(force_redownload=True)
        validate_tarball(TAR_PATH)

    extract_tarball()
    return locate_fgvc_data_dir(EXTRACT_DIR)


# Resolve the final dataset directories used by the rest of the notebook.
data_dir = ensure_fgvc_ready()
images_dir = data_dir / "images"

print("FGVC data directory:", data_dir)
print("Raw images directory:", images_dir, "| exists:", images_dir.exists())
print("Preprocessed images directory:", PROC_IMAGES_DIR)
print("Model export path:", MODEL_PATH)


## Load split annotations and aircraft bounding boxes

This section loads:
- the official manufacturer labels for train / validation / test,
- the aircraft bounding boxes from `images_box.txt`.

The bounding boxes are essential because they let the notebook focus on the aircraft instead of the full background.
This is especially important in a fine-grained classification setting, where small visual details matter.


In [ ]:
from pathlib import Path
import pandas as pd

# -----------------------------------------------------------------------------
# HELPERS TO LOAD THE OFFICIAL SPLIT FILES
# -----------------------------------------------------------------------------
def must_exist(p: Path) -> Path:
    """Raise a clear error if a required file is missing."""
    if not p.exists():
        raise FileNotFoundError(str(p))
    return p


# Official manufacturer split files.
train_file = must_exist(data_dir / "images_manufacturer_train.txt")
val_file   = must_exist(data_dir / "images_manufacturer_val.txt")
test_file  = must_exist(data_dir / "images_manufacturer_test.txt")

# Official bounding-box annotations.
bbox_file  = must_exist(data_dir / "images_box.txt")


def read_split(txt_path: Path):
    """Read an FGVC split file where labels may contain spaces.

    Each line has the form:
        image_id label

    The label may contain several words, so a simple fixed-width split would fail.
    The parser therefore reads:
    - the first token as image_id,
    - the rest of the line as the textual label.
    """
    rows = []

    with open(txt_path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(f"Malformed line {ln} in {txt_path}: {line!r}")

            image_id, label = parts[0], parts[1].strip()
            rows.append((image_id, label))

    return pd.DataFrame(rows, columns=["image_id", "label"])


# Load the official manufacturer splits.
df_train = read_split(train_file)
df_val   = read_split(val_file)
df_test  = read_split(test_file)

# Load the official bounding boxes.
# Format: image_id xmin ymin xmax ymax
bbox_df = pd.read_csv(
    bbox_file,
    sep=r"\s+",
    header=None,
    names=["image_id", "xmin", "ymin", "xmax", "ymax"],
    engine="python"
)
bbox_df["image_id"] = bbox_df["image_id"].astype(str)

print("Train samples:", len(df_train))
print("Validation samples:", len(df_val))
print("Test samples:", len(df_test))
print("Bounding boxes:", len(bbox_df))

df_train.head()


## Offline image preprocessing

Each image is preprocessed **once** and saved to disk.

The preprocessing pipeline is:
1. remove the 20-pixel copyright banner,
2. crop the aircraft using the official bounding box,
3. add a small margin around the aircraft,
4. save the processed crop in a cache folder.

This offline cache makes the whole training pipeline faster and more stable because the expensive image operations are not repeated every epoch.
It also makes the preprocessing logic explicit and reproducible.


In [ ]:
from PIL import Image
from tqdm.auto import tqdm
import numpy as np

# -----------------------------------------------------------------------------
# PREPROCESSING LOGIC
# -----------------------------------------------------------------------------
# Build a dictionary for fast image_id -> bounding box lookup.
bbox_map = {
    row.image_id: (int(row.xmin), int(row.ymin), int(row.xmax), int(row.ymax))
    for row in bbox_df.itertuples(index=False)
}

# Collect every image id across train / validation / test.
# This ensures that a single cached preprocessing pass is enough for the whole notebook.
ALL_IDS = pd.concat([df_train["image_id"], df_val["image_id"], df_test["image_id"]]).unique().tolist()


def crop_and_save(image_id: str, margin: float = 0.08):
    """Preprocess one image and save the result.

    Steps:
    1. load the original JPEG,
    2. remove the 20-pixel bottom copyright banner,
    3. crop the aircraft using the official bounding box,
    4. enlarge the crop slightly with a margin,
    5. save the processed image.

    Parameters
    ----------
    image_id : str
        FGVC image identifier without the .jpg suffix.
    margin : float
        Relative margin added around the bounding box.
    """
    src = images_dir / f"{image_id}.jpg"
    dst = PROC_IMAGES_DIR / f"{image_id}.jpg"

    # Skip work when the processed file already exists.
    if dst.exists():
        return

    # Load RGB image.
    im = Image.open(src).convert("RGB")
    w, h = im.size

    # Remove the copyright banner.
    # The dataset documentation explicitly states that the bottom 20 pixels
    # contain copyright information and should be removed.
    h2 = max(1, h - 20)
    im = im.crop((0, 0, w, h2))
    w, h = im.size

    # Load the official bounding box for this image.
    # The original annotations are 1-indexed.
    xmin, ymin, xmax, ymax = bbox_map.get(image_id, (1, 1, w, h))

    # Convert to 0-indexed coordinates and clamp them to the image dimensions.
    xmin = max(0, xmin - 1)
    ymin = max(0, ymin - 1)
    xmax = min(w, xmax)
    ymax = min(h, ymax)

    # Compute box width and height.
    bw = max(1, xmax - xmin)
    bh = max(1, ymax - ymin)

    # Add a small margin around the aircraft to preserve some context.
    dx = int(bw * margin)
    dy = int(bh * margin)

    x0 = max(0, xmin - dx)
    y0 = max(0, ymin - dy)
    x1 = min(w, xmax + dx)
    y1 = min(h, ymax + dy)

    # Final aircraft-centered crop.
    im = im.crop((x0, y0, x1, y1))

    # Save to cache.
    im.save(dst, quality=95, optimize=True)


# Process only the images that are still missing in the cache.
missing = [iid for iid in ALL_IDS if not (PROC_IMAGES_DIR / f"{iid}.jpg").exists()]
print("Missing preprocessed images:", len(missing), "out of", len(ALL_IDS))

for iid in tqdm(missing):
    crop_and_save(iid)

# Quick cache sanity check.
print("Processed images available:", len(list(PROC_IMAGES_DIR.glob("*.jpg"))))
print("Example processed file exists:", (PROC_IMAGES_DIR / f"{ALL_IDS[0]}.jpg").exists())


## Build FastAI DataLoaders

This section creates DataLoaders from the preprocessed aircraft crops.

Main design choices:
- images are resized to a fixed square shape **before batching**,
- padding is used to preserve aspect ratio,
- data augmentation is applied during training,
- the official validation split is used explicitly.

The validation set remains separate because it is still needed to monitor the training behavior.


In [ ]:
from fastai.vision.all import *
from fastai.callback.mixup import MixUp

# -----------------------------------------------------------------------------
# DATALOADER CONFIGURATION
# -----------------------------------------------------------------------------
# Ensure reproducibility of the training pipeline as much as possible.
set_seed(42, reproducible=True)

# Combine train and validation into one dataframe while preserving the official
# validation indices. FastAI will use valid_idx to split them internally.
df_all = pd.concat([df_train, df_val], ignore_index=True)
valid_idx = list(range(len(df_train), len(df_train) + len(df_val)))

# Fixed backbone required for this notebook.
BACKBONE = resnet34


def make_dls(img_size: int, bs: int = 64):
    """Build FastAI DataLoaders for a given image size and batch size.

    Parameters
    ----------
    img_size : int
        Final square image size before batching.
    bs : int
        Batch size.

    Returns
    -------
    DataLoaders
        FastAI DataLoaders object.
    """

    # Resize happens before batching, which guarantees a consistent tensor shape.
    # Padding preserves the aspect ratio and avoids geometric distortion.
    item_tfms = Resize((img_size, img_size), method=ResizeMethod.Pad, pad_mode='zeros')

    # Batch transforms include standard training augmentations and ImageNet normalization.
    batch_tfms = [
        *aug_transforms(min_scale=0.75),
        Normalize.from_stats(*imagenet_stats)
    ]

    dls = ImageDataLoaders.from_df(
        df_all,
        path=PROC_IMAGES_DIR,
        fn_col="image_id",
        suff=".jpg",
        label_col="label",
        valid_idx=valid_idx,
        item_tfms=item_tfms,
        batch_tfms=batch_tfms,
        bs=bs,
        num_workers=2
    )

    return dls


# First DataLoaders instance used for the initial stage of progressive resizing.
dls_256 = make_dls(256, bs=96)
dls_256.show_batch(max_n=9)


## Build the learner

The learner includes:
- a **ResNet34** backbone,
- weight decay,
- **MixUp** regularization,
- mixed precision when available,
- a workaround for FastAI progress-bar issues in some Kaggle notebook environments.

This keeps the training loop robust while remaining close to a standard FastAI workflow.


In [ ]:
from fastai.vision.all import *
from fastai.callback.progress import ProgressCallback, ShowGraphCallback

# -----------------------------------------------------------------------------
# LEARNER FACTORY
# -----------------------------------------------------------------------------
def build_learner(dls):
    """Create a FastAI learner configured for this experiment.

    The learner uses:
    - ResNet34,
    - accuracy as the live validation metric,
    - weight decay,
    - MixUp as a regularization callback,
    - mixed precision when supported.
    """

    # MixUp can be helpful in fine-grained classification by making the model
    # less brittle and improving generalization.
    cbs = [MixUp(alpha=0.2)]

    learn = vision_learner(
        dls,
        BACKBONE,
        metrics=[accuracy],
        wd=1e-2,
        cbs=cbs
    )

    # Some Kaggle notebook environments can fail with FastAI's notebook progress
    # bars. Removing these callbacks makes the training loop more reliable.
    try:
        learn.remove_cb(ProgressCallback)
    except Exception:
        pass

    try:
        learn.remove_cb(ShowGraphCallback)
    except Exception:
        pass

    # Mixed precision reduces memory usage and can speed up training.
    try:
        learn = learn.to_fp16()
    except Exception:
        pass

    return learn


learn = build_learner(dls_256)
learn


## Progressive training overview

The model is trained in three stages:
1. **256×256**
2. **384×384**
3. **512×512**

This strategy is known as **progressive resizing**.
It usually works well in fine-grained classification because the model first learns global structure cheaply, then progressively incorporates more detail at higher resolutions.


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 1 — 256x256
# -----------------------------------------------------------------------------
# This first stage is the fastest one. It gives the model a strong initialization
# before moving to larger and more detailed images.
learn.fine_tune(6, base_lr=3e-3)

# Validation metrics after stage 1.
learn.validate()


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 2 — 384x384
# -----------------------------------------------------------------------------
# The weights learned at 256x256 are kept. Only the DataLoaders are rebuilt with
# a higher image resolution, and the learner continues from the previous state.
dls_384 = make_dls(384, bs=64)
learn.dls = dls_384

learn.fine_tune(4, base_lr=2e-3)

# Validation metrics after stage 2.
learn.validate()


In [ ]:
# -----------------------------------------------------------------------------
# STAGE 3 — 512x512
# -----------------------------------------------------------------------------
# This final stage is the slowest but often the most informative for a
# fine-grained recognition problem, where small visual details matter.
dls_512 = make_dls(512, bs=32)
learn.dls = dls_512

learn.fine_tune(3, base_lr=1e-3)

# Validation metrics after stage 3.
learn.validate()


## Prepare report folders

All evaluation artifacts are exported in a dedicated folder structure so they can be downloaded and reused outside the notebook.


In [ ]:
from pathlib import Path

# -----------------------------------------------------------------------------
# REPORT OUTPUT STRUCTURE
# -----------------------------------------------------------------------------
REPORT_DIR = OUTPUT_DIR / "reports"
PLOT_DIR = REPORT_DIR / "plots"
TABLE_DIR = REPORT_DIR / "tables"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR, PLOT_DIR, TABLE_DIR


## Full evaluation on the official test split

This is the final and most important evaluation block.

The model has already been trained and selected.
It is now evaluated on the **official test split only**.

The notebook computes:
- accuracy
- balanced accuracy / macro recall
- macro precision
- macro F1-score
- top-3 accuracy

Balanced accuracy is especially important here because the FGVC benchmark is fundamentally based on the diagonal of the row-normalized confusion matrix, which aligns closely with macro recall.


In [ ]:
from fastai.vision.all import *
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    top_k_accuracy_score,
)
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# TEST-SPLIT EVALUATION
# -----------------------------------------------------------------------------
# Use the final image size from the last stage of progressive resizing.
FINAL_IMG_SIZE = 512
FINAL_BS = 32


def make_test_dl(img_size: int = 512, bs: int = 32):
    """Build a deterministic DataLoader for test-time evaluation.

    No training augmentation is applied here.
    Only resizing and ImageNet normalization are kept.
    """
    item_tfms = Resize((img_size, img_size), method=ResizeMethod.Pad, pad_mode='zeros')
    batch_tfms = [Normalize.from_stats(*imagenet_stats)]

    test_dl = ImageDataLoaders.from_df(
        df_test,
        path=PROC_IMAGES_DIR,
        fn_col="image_id",
        suff=".jpg",
        label_col="label",
        valid_pct=0.0,
        item_tfms=item_tfms,
        batch_tfms=batch_tfms,
        bs=bs,
        shuffle=False,
        num_workers=2
    ).train

    return test_dl


# Build the official test DataLoader.
test_dl = make_test_dl(FINAL_IMG_SIZE, FINAL_BS)

# Collect predicted probabilities and target indices.
probs, targs = learn.get_preds(dl=test_dl, with_input=False, with_decoded=False)
pred_idx = probs.argmax(dim=1)

# Recover the vocabulary used by the learner.
vocab = list(learn.dls.vocab)
idx_to_label = {i: lbl for i, lbl in enumerate(vocab)}

# Convert tensors to NumPy arrays for sklearn metrics.
y_true = targs.cpu().numpy()
y_pred = pred_idx.cpu().numpy()
probs_np = probs.cpu().numpy()

# Human-readable labels.
true_labels = [idx_to_label[int(i)] for i in y_true]
pred_labels = [idx_to_label[int(i)] for i in y_pred]

# Macro metrics.
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

# Summary table with the key evaluation metrics.
metrics_summary = pd.DataFrame([{
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy_fgvc": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "top3_accuracy": float(top_k_accuracy_score(y_true, probs_np, k=3, labels=np.arange(len(vocab))))
}])

metrics_summary


## Test-time augmentation evaluation

This second evaluation pass measures the model again with **test-time augmentation (TTA)**.

TTA applies several deterministic or lightly augmented views of each image at inference time and averages the predictions.
This is useful because:
- it often improves robustness,
- it provides a direct comparison with the standard evaluation,
- it helps determine whether the model is sensitive to small input variations.


In [ ]:
# -----------------------------------------------------------------------------
# TEST-TIME AUGMENTATION EVALUATION
# -----------------------------------------------------------------------------
# Rebuild a deterministic test DataLoader and run TTA on it.
test_dl_tta = make_test_dl(FINAL_IMG_SIZE, FINAL_BS)
probs_tta, targs_tta = learn.tta(dl=test_dl_tta)

pred_idx_tta = probs_tta.argmax(dim=1)
y_pred_tta = pred_idx_tta.cpu().numpy()

macro_precision_tta, macro_recall_tta, macro_f1_tta, _ = precision_recall_fscore_support(
    y_true,
    y_pred_tta,
    average="macro",
    zero_division=0
)

metrics_summary_tta = pd.DataFrame([{
    "accuracy": float(accuracy_score(y_true, y_pred_tta)),
    "balanced_accuracy_fgvc": float(balanced_accuracy_score(y_true, y_pred_tta)),
    "macro_precision": float(macro_precision_tta),
    "macro_recall": float(macro_recall_tta),
    "macro_f1": float(macro_f1_tta),
    "top3_accuracy": float(top_k_accuracy_score(y_true, probs_tta.cpu().numpy(), k=3, labels=np.arange(len(vocab))))
}])

metrics_summary_tta


## Per-image prediction table

This table stores the prediction for every test image, including:
- the true class,
- the predicted class,
- a correctness flag,
- the top predicted confidence,
- the top-5 predicted classes and probabilities.

This is useful for manual inspection of errors and difficult cases.


In [ ]:
# -----------------------------------------------------------------------------
# PER-IMAGE PREDICTION TABLE
# -----------------------------------------------------------------------------
topk = min(5, len(vocab))
top_vals, top_idxs = torch.topk(probs, k=topk, dim=1)

rows = []
for i, image_id in enumerate(df_test["image_id"].tolist()):
    row = {
        "image_id": image_id,
        "true_label": true_labels[i],
        "pred_label": pred_labels[i],
        "is_correct": bool(y_true[i] == y_pred[i]),
        "pred_confidence": float(top_vals[i, 0].item()),
    }

    # Store the complete top-k prediction profile.
    for k in range(topk):
        row[f"top{k+1}_label"] = idx_to_label[int(top_idxs[i, k].item())]
        row[f"top{k+1}_prob"] = float(top_vals[i, k].item())

    rows.append(row)

predictions_df = pd.DataFrame(rows)
predictions_df.head()


## Per-class performance report

This section transforms the global test results into a **manufacturer-by-manufacturer diagnostic**.

For each class, it reports:
- precision
- recall
- F1-score
- support

For this problem, recall is especially informative because it corresponds to the class-wise diagonal term of the row-normalized confusion matrix.


In [ ]:
# -----------------------------------------------------------------------------
# PER-CLASS REPORT
# -----------------------------------------------------------------------------
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=vocab,
    output_dict=True,
    zero_division=0
)

per_class_df = (
    pd.DataFrame(report_dict)
    .T
    .reset_index()
    .rename(columns={"index": "label"})
)

# Keep only true class rows and remove global summary rows such as accuracy/macro avg.
per_class_df = per_class_df[per_class_df["label"].isin(vocab)].copy()
per_class_df["support"] = per_class_df["support"].astype(int)

# Rename recall to emphasize its interpretation as class accuracy.
per_class_df = per_class_df.rename(columns={
    "precision": "precision",
    "recall": "class_accuracy_recall",
    "f1-score": "f1_score",
})

# Identify the strongest and weakest classes.
worst_classes = per_class_df.sort_values(["class_accuracy_recall", "support"], ascending=[True, False]).head(15)
best_classes  = per_class_df.sort_values(["class_accuracy_recall", "support"], ascending=[False, False]).head(15)

worst_classes


## Confusion matrix

The confusion matrix is computed and then **row-normalized**.

This representation is especially useful because:
- rows correspond to the true class,
- columns correspond to the predicted class,
- the diagonal gives class-wise accuracy,
- off-diagonal values reveal systematic confusions.

A row-normalized matrix is much easier to interpret than raw counts when classes have different support.


In [ ]:
# -----------------------------------------------------------------------------
# CONFUSION MATRIX
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(vocab)))

# Normalize each row independently so that each row sums to 1.
cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig = plt.figure(figsize=(16, 14))
plt.imshow(cm_norm, interpolation="nearest")
plt.title("FGVC-Aircraft Manufacturer — Row-normalized confusion matrix (test)")
plt.colorbar()
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()

cm_plot_path = PLOT_DIR / "confusion_matrix_row_normalized_test.png"
plt.savefig(cm_plot_path, dpi=200, bbox_inches="tight")
plt.show()

cm_plot_path


## Largest confusion pairs

This section extracts the most important non-diagonal errors from the confusion matrix.

It helps answer practical questions such as:
- which manufacturers are most often confused with one another?
- are these mistakes isolated or systematic?
- which confusions matter the most in raw counts and in row-normalized terms?


In [ ]:
# -----------------------------------------------------------------------------
# TOP CONFUSION PAIRS
# -----------------------------------------------------------------------------
confusions = []

for i, true_lbl in enumerate(vocab):
    for j, pred_lbl in enumerate(vocab):
        if i == j:
            continue

        count = int(cm[i, j])
        if count > 0:
            confusions.append({
                "true_label": true_lbl,
                "pred_label": pred_lbl,
                "count": count,
                "row_normalized_rate": float(cm_norm[i, j]),
            })

top_confusions_df = pd.DataFrame(confusions).sort_values(
    ["count", "row_normalized_rate"],
    ascending=[False, False]
).reset_index(drop=True)

top_confusions_df.head(25)


## Export evaluation artifacts

All key outputs are exported to files:
- summary metrics
- TTA summary metrics
- per-image predictions
- per-class report
- strongest and weakest classes
- top confusion pairs
- confusion matrix plot

This ensures that the notebook produces reusable artifacts, not just in-memory results.


In [ ]:
# -----------------------------------------------------------------------------
# EXPORT EVALUATION ARTIFACTS
# -----------------------------------------------------------------------------
metrics_path = TABLE_DIR / "test_metrics_summary.csv"
metrics_tta_path = TABLE_DIR / "test_metrics_summary_tta.csv"
preds_path = TABLE_DIR / "test_predictions_top5.csv"
per_class_path = TABLE_DIR / "test_per_class_report.csv"
worst_classes_path = TABLE_DIR / "test_worst_classes.csv"
best_classes_path = TABLE_DIR / "test_best_classes.csv"
top_confusions_path = TABLE_DIR / "test_top_confusions.csv"

metrics_summary.to_csv(metrics_path, index=False)
metrics_summary_tta.to_csv(metrics_tta_path, index=False)
predictions_df.to_csv(preds_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
worst_classes.to_csv(worst_classes_path, index=False)
best_classes.to_csv(best_classes_path, index=False)
top_confusions_df.to_csv(top_confusions_path, index=False)

print("Saved:")
print(metrics_path)
print(metrics_tta_path)
print(preds_path)
print(per_class_path)
print(worst_classes_path)
print(best_classes_path)
print(top_confusions_path)
print(cm_plot_path)


## Train- versus test-behavior interpretation

At this stage, the notebook provides all the information needed to diagnose model behavior:
- whether validation performance transfers to test performance,
- which manufacturers are reliably recognized,
- which manufacturers are unstable,
- whether TTA improves robustness,
- which specific confusion pairs dominate the error profile.

This diagnostic stage is essential before changing the backbone, merging splits, or deploying a new version.


## Export the FastAI learner

The FastAI learner is exported as a `.pkl` file.
This is convenient for experimentation and for reuse in an environment compatible with the same FastAI stack.


In [ ]:
# -----------------------------------------------------------------------------
# FASTAI LEARNER EXPORT
# -----------------------------------------------------------------------------
learn.export(MODEL_PATH)
MODEL_PATH


## Export portable deployment artifacts

FastAI pickle files are convenient, but deployment is usually safer with more explicit artifacts.

For that reason, the notebook also exports:
- `model.pth` containing the PyTorch weights,
- `labels.json` containing the exact class order.

This is especially useful when moving from Kaggle to a local machine or a Hugging Face Space.


In [ ]:
# -----------------------------------------------------------------------------
# PORTABLE DEPLOYMENT EXPORTS
# -----------------------------------------------------------------------------
import json
import torch

# Save raw PyTorch weights.
pth_path = MODEL_DIR / "model.pth"
torch.save(learn.model.state_dict(), pth_path)

# Save the exact label order used by the learner.
labels_path = MODEL_DIR / "labels.json"
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(list(learn.dls.vocab), f, ensure_ascii=False, indent=2)

print("Portable artifacts saved:")
print(pth_path)
print(labels_path)


## Quick sanity check on random test images

The notebook ends with a simple qualitative sanity check:
a few random test images are passed through the exported learner to verify that prediction calls still work as expected.


In [ ]:
# -----------------------------------------------------------------------------
# SANITY CHECK ON RANDOM TEST IMAGES
# -----------------------------------------------------------------------------
learn_inf = load_learner(MODEL_PATH)

for image_id in df_test.sample(8, random_state=42)["image_id"].tolist():
    p = PROC_IMAGES_DIR / f"{image_id}.jpg"
    pred, pred_idx, probs = learn_inf.predict(p)
    print(p.name, "->", pred)
